# 🏦 Exercícios SQL — CVM Data Lake
### Versão do Aluno | Big Data for Finance — FAE
**Banco:** PostgreSQL (DBeaver local)

**Tabelas utilizadas:**
- `layer_02_silver` — Tabelas que estão na camada Silver

**CNPJs de referência rápida:**
| Empresa | CNPJ |
|---|---|
| Petrobras | `33.000.167/0001-01` |
| Ambev | `07.526.557/0001-00` |
| Vale | `33.592.510/0001-54` |

---

## ⚙️ Configuração da Conexão

In [3]:
from dotenv import load_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine
import os

# 1. Carrega o arquivo .env (2 níveis acima se estiver em notebooks/01_bronze/)
load_dotenv('../../.env')

# 2. Lê as variáveis de ambiente
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# 3. Encoding de segurança: transforma 'senha@123' em 'senha%40123'
encoded_pass = quote_plus(DB_PASS)

# 4. Monta e cria a engine SQLAlchemy
conn_string = f"postgresql+psycopg2://{DB_USER}:{encoded_pass}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_string)

# ... (mantenha seu código de carregamento do .env e criação da engine)

# ADICIONE ESTA FUNÇÃO NO FINAL DA CÉLULA 1:
def run_query(query):
    import pandas as pd
    return pd.read_sql(query, engine)

print("✅ Conexão estabelecida e função run_query pronta!")

✅ Conexão estabelecida e função run_query pronta!


---
## Questão 1 — Primeira olhada na tabela Silver 🟢 Fácil
**Contexto:** A primeira coisa que um analista faz ao chegar em um novo banco é explorar o que existe dentro da tabela. A camada Silver já tem os dados do Balanço Patrimonial limpos, normalizados e prontos para análise.

**Pergunta:** Mostre as **10 primeiras linhas** da tabela `n1_dfp_cia_aberta_bp`, exibindo apenas as colunas: `CNPJ_CIA`, `DENOM_CIA`, `DT_REFER`, `CD_CONTA`, `DS_CONTA`, `VL_CONTA_TRATADO` e `IS_LEAF`.

In [4]:
query = """
SELECT "CNPJ_CIA", "DENOM_CIA", "DT_REFER", "CD_CONTA", "DS_CONTA", "VL_CONTA_TRATADO", "IS_LEAF"
FROM layer_02_silver.n1_dfp_cia_aberta_bp
LIMIT 10;
"""
run_query(query)

,CNPJ_CIA,DENOM_CIA,DT_REFER,CD_CONTA,DS_CONTA,VL_CONTA_TRATADO,IS_LEAF
0,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1,ATIVO TOTAL,2.119934e+09,False
1,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01,ATIVO CIRCULANTE,4.499600e+08,False
2,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.01,CAIXA E EQUIVALENTES DE CAIXA,9.925800e+07,False
3,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.03,CONTAS A RECEBER,3.211700e+08,False
4,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.03.01,CLIENTES,3.211700e+08,False
5,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.03.01.01,CLIENTES,3.059410e+08,True
6,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.03.01.09,(-) PROVISÃO P/CRÉDITO DE LIQUIDAÇÃO DUVIDOSA,1.522900e+07,True
7,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.04,ESTOQUES,8.619000e+06,False
8,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.06,TRIBUTOS A RECUPERAR,2.091300e+07,False
9,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.06.01,TRIBUTOS CORRENTES A RECUPERAR,2.091300e+07,False


---
## Questão 2 — Volume e cobertura da base 🟢 Fácil
**Contexto:** Antes de qualquer análise, o analista precisa entender o tamanho e a cobertura da base: quantas linhas existem, quantas empresas distintas estão cobertas e qual o intervalo de datas disponível.

**Pergunta:** Em uma única query, calcule para a tabela `n1_dfp_cia_aberta_bp`:
- Total de linhas → `total_linhas`
- Quantidade de empresas distintas (por `CNPJ_CIA`) → `empresas_distintas`
- Quantidade de datas de referência distintas (`DT_REFER`) → `datas_distintas`

In [5]:
query = """
SELECT 
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT "CNPJ_CIA") AS empresas_distintas,
    COUNT(DISTINCT "DT_REFER") AS datas_distintas
FROM layer_02_silver.n1_dfp_cia_aberta_bp;
"""
run_query(query)

,total_linhas,empresas_distintas,datas_distintas
0,180461,185,54


---
## Questão 3 — Distribuição de empresas por setor 🟢 Fácil
**Contexto:** A CVM classifica cada empresa listada em um setor de atividade (`SETOR_ATIV`). Entender essa distribuição é o ponto de partida para qualquer análise setorial comparativa. Use a tabela `n0_empresas_selecionadas`, que contém o cadastro completo das empresas do nosso universo.

**Pergunta:** Quantas empresas existem em cada `SETOR_ATIV`? Ordene do setor com **mais** empresas para o com **menos**.

In [6]:
query = """
SELECT 
    "SETOR_ATIV", 
    COUNT(*) AS qtd_empresas
FROM layer_02_silver.n0_empresas_selecionadas
GROUP BY "SETOR_ATIV"
ORDER BY qtd_empresas DESC;
"""
run_query(query)

,SETOR_ATIV,qtd_empresas
0,Energia Elétrica,33
1,"Construção Civil, Mat. Constr. e Decoração",30
2,Comércio (Atacado e Varejo),24
3,Serviços Transporte e Logística,17
4,Metalurgia e Siderurgia,13
5,Têxtil e Vestuário,13
6,"Máquinas, Equipamentos, Veículos e Peças",13
7,Alimentos,9
8,"Saneamento, Serv. Água e Gás",8
9,Serviços Médicos,8


---
## Questão 4 — Petrobras: Ativo e Passivo mais recentes 🟢 Fácil
**Contexto:** No Balanço Patrimonial da CVM, `CD_CONTA = '1'` é o **Ativo Total** e `CD_CONTA = '2'` é o **Passivo + Patrimônio Líquido Total**. A equação fundamental do BP é: **Ativo = Passivo + PL** — qualquer diferença indica erro de dados.

**Pergunta:** Para a **Petrobras** (`CNPJ_CIA = '33.000.167/0001-01'`), mostre o valor (em bilhões de reais) das contas `'1'` e `'2'` para a **data de referência mais recente** disponível. Use uma subconsulta escalar com `MAX("DT_REFER")` para encontrar a data mais recente.

*Dica: `ROUND((valor / 1e9)::NUMERIC, 2)` converte para bilhões com 2 casas.*

In [7]:
query = """
SELECT 
    "DS_CONTA",
    ROUND(("VL_CONTA_TRATADO" / 1e9)::NUMERIC, 2) AS valor_bilhoes
FROM layer_02_silver.n1_dfp_cia_aberta_bp
WHERE "CNPJ_CIA" = '33.000.167/0001-01'
  AND "CD_CONTA" IN ('1', '2')
  AND "DT_REFER" = (
      SELECT MAX("DT_REFER") 
      FROM layer_02_silver.n1_dfp_cia_aberta_bp 
      WHERE "CNPJ_CIA" = '33.000.167/0001-01'
  );
"""
run_query(query)

,DS_CONTA,valor_bilhoes
0,ATIVO TOTAL,1124.8
1,PASSIVO TOTAL,1124.8


---
## Questão 5 — IS_LEAF: folhas vs. agregadores 🟢 Fácil
**Contexto:** `IS_LEAF = TRUE` marca contas **analíticas** (sem filhos) — são as únicas que devem ser somadas em qualquer ferramenta de BI, pois evitam dupla contagem. `IS_LEAF = FALSE` são contas **sintéticas** (pais) que já incorporam os valores de todas as suas contas filhas. Somar tudo geraria dupla contagem.

**Pergunta:** Para a **Petrobras** (`CNPJ_CIA = '33.000.167/0001-01'`) em `DT_REFER = '2023-12-31'`, quantas contas são `IS_LEAF = TRUE` e quantas são `IS_LEAF = FALSE`? Use `GROUP BY "IS_LEAF"`.

In [10]:
query = """
SELECT 
    "IS_LEAF", 
    COUNT(*) AS total_contas
FROM layer_02_silver.n1_dfp_cia_aberta_bp
WHERE "CNPJ_CIA" = '33.000.167/0001-01'
  AND "DT_REFER" = '2023-12-31'
GROUP BY "IS_LEAF";
"""
run_query(query)

,IS_LEAF,total_contas
0,False,51
1,True,19


---
## Questão 6 — Top 10 maiores empresas por Ativo Total em 2023 🟡 Médio
**Contexto:** Rankings de tamanho de balanço são análises clássicas de mercado de capitais. O Ativo Total (`CD_CONTA = '1'`) resume o tamanho total de uma empresa — tudo que ela possui e tem direito a receber.

**Pergunta:** Liste as **10 empresas com maior Ativo Total** em `DT_REFER = '2023-12-31'`. Exiba `DENOM_CIA`, `SETOR_ATIV` e o valor em **bilhões** (2 casas decimais). Ordene do maior para o menor.

*Dica: `ROUND((expr)::NUMERIC, 2)` — o cast para `NUMERIC` é obrigatório no PostgreSQL quando a coluna é `DOUBLE PRECISION`.*

In [8]:
query = """
SELECT 
    t1."DENOM_CIA", 
    t2."SETOR_ATIV",
    ROUND((t1."VL_CONTA_TRATADO" / 1e9)::NUMERIC, 2) AS ativo_total_bilhoes
FROM layer_02_silver.n1_dfp_cia_aberta_bp t1
JOIN layer_02_silver.n0_empresas_selecionadas t2 ON t1."CNPJ_CIA" = t2."CNPJ_CIA"
WHERE t1."DT_REFER" = '2023-12-31'
  AND t1."CD_CONTA" = '1'
ORDER BY t1."VL_CONTA_TRATADO" DESC
LIMIT 10;
"""
run_query(query)

,DENOM_CIA,SETOR_ATIV,ativo_total_bilhoes
0,PETROLEO BRASILEIRO S.A. PETROBRAS,Petróleo e Gás,1050.89
1,VALE S.A.,Extração Mineral,455.98
2,JBS S.A.,Alimentos,206.13
3,SUZANO S.A.,Papel e Celulose,143.59
4,COSAN S.A.,"Agricultura (Açúcar, Álcool e Cana)",139.87
5,AMBEV S.A.,Bebidas e Fumo,132.64
6,MARFRIG GLOBAL FOODS S.A.,Alimentos,130.95
7,TELEFÔNICA BRASIL S.A,Telecomunicações,120.74
8,EQUATORIAL S.A.,Energia Elétrica,103.64
9,BRASKEM S.A.,Petroquímicos e Borracha,91.74


---
## Questão 7 — Evolução anual do Ativo Total da Ambev 🟡 Médio
**Contexto:** Acompanhar a evolução do balanço ao longo dos anos revela a trajetória de crescimento (ou retração) de uma empresa. `DT_REFER` é do tipo `TIMESTAMP` — use `EXTRACT(YEAR FROM ...)` para extrair o ano.

**Pergunta:** Para a **Ambev** (`CNPJ_CIA = '07.526.557/0001-00'`), mostre a evolução anual do **Ativo Total** (`CD_CONTA = '1'`). Extraia o ano de `DT_REFER`, exiba o valor em bilhões e ordene do ano mais antigo ao mais recente.

In [11]:
query = """
SELECT 
    EXTRACT(YEAR FROM "DT_REFER") AS ano,
    ROUND(("VL_CONTA_TRATADO" / 1e9)::NUMERIC, 2) AS ativo_total_bilhoes
FROM layer_02_silver.n1_dfp_cia_aberta_bp
WHERE "CNPJ_CIA" = '07.526.557/0001-00'
  AND "CD_CONTA" = '1'
ORDER BY ano ASC;
"""
run_query(query)

,ano,ativo_total_bilhoes
0,2013.0,68.67
1,2014.0,72.14
2,2015.0,90.18
3,2016.0,83.84
4,2017.0,86.85
5,2018.0,94.13
6,2019.0,101.74
7,2020.0,125.20
8,2021.0,138.60
9,2022.0,137.96


---
## Questão 8 — Contas reconstruídas pelo pipeline 🟡 Médio
**Contexto:** `FLAG_RECONSTRUCAO = TRUE` indica que uma linha foi criada **sinteticamente** pelo pipeline — a empresa não reportou aquela conta e o pipeline reconstruiu o valor somando os filhos. Mais contas reconstruídas = mais "buracos" no dado original enviado à CVM.

**Pergunta:** Em **toda a base histórica**, quais são as **10 combinações de empresa e data** com mais contas reconstruídas no BP? Mostre `DT_REFER`, `DENOM_CIA` e o total de contas de cada empresa. Use `SUM(CASE WHEN "FLAG_RECONSTRUCAO" = TRUE THEN 1 ELSE 0 END)` para contar. Agrupe por `"DT_REFER"` e `"DENOM_CIA"`. Ordene da mais para a menos reconstruída.


In [12]:
query = """
SELECT 
    "DT_REFER", 
    "DENOM_CIA", 
    SUM(CASE WHEN "FLAG_RECONSTRUCAO" = TRUE THEN 1 ELSE 0 END) AS total_reconstruidas
FROM layer_02_silver.n1_dfp_cia_aberta_bp
GROUP BY "DT_REFER", "DENOM_CIA"
ORDER BY total_reconstruidas DESC
LIMIT 10;
"""
run_query(query)

,DT_REFER,DENOM_CIA,total_reconstruidas
0,2022-12-31,CESP - CIA ENERGETICA DE SAO PAULO,15
1,2021-12-31,VIVER INCORPORADORA E CONSTRUTORA S.A.,2
2,2017-12-31,CIA SIDERURGICA NACIONAL,1
3,2010-12-31,USINAS SID DE MINAS GERAIS S.A.-USIMINAS,1
4,2016-12-31,CIA SIDERURGICA NACIONAL,1
5,2021-12-31,INTERNATIONAL MEAL COMPANY ALIMENTACAO S.A.,1
6,2022-12-31,GRUPO CASAS BAHIA S.A.,1
7,2022-12-31,CIA BRASILEIRA DE DISTRIBUICAO,1
8,2021-12-31,CVC BRASIL OPERADORA E AGÊNCIA DE VIAGENS S.A.,1
9,2018-12-31,PRIO S.A.,0


---
## Questão 9 — Base analítica de Receita Bruta por empresa via JOIN 🟡 Médio
**Contexto:** Antes de agregar, o analista precisa da base analítica: uma linha por empresa com seu setor e receita. Com esse DataFrame em mãos, o Pandas consegue gerar estatísticas descritivas por grupo com `.describe()`.

**Pergunta:** Faça um `JOIN` entre `n1_dfp_cia_aberta_dre` e `n0_empresas_selecionadas`. Para `DT_REFER = '2023-12-31'` e `CD_CONTA = '3.01'` (Receita Bruta), traga uma linha por empresa com `CNPJ_CIA`, `DENOM_CIA`, `SETOR_ATIV` e o valor em bilhões. Depois use `.groupby` + `.describe()` para ver as estatísticas por setor.


In [9]:
query = """
SELECT 
    dre."CNPJ_CIA", 
    dre."DENOM_CIA", 
    emp."SETOR_ATIV", 
    (dre."VL_CONTA_TRATADO" / 1e9) AS receita_bilhoes
FROM layer_02_silver.n1_dfp_cia_aberta_dre dre
JOIN layer_02_silver.n0_empresas_selecionadas emp ON dre."CNPJ_CIA" = emp."CNPJ_CIA"
WHERE dre."DT_REFER" = '2023-12-31'
  AND dre."CD_CONTA" = '3.01';
"""
df_receita = run_query(query)

# Análise estatística por setor
df_receita.groupby("SETOR_ATIV")["receita_bilhoes"].describe()

,count,mean,std,min,25%,50%,75%,max
SETOR_ATIV,,,,,,,,
"Agricultura (Açúcar, Álcool e Cana)",3.0,16.256558,20.593450,0.179054,4.650589,9.122124,24.295310,39.468497
Alimentos,7.0,77.230353,134.754948,0.777726,3.034153,10.840280,79.554810,363.816537
Bebidas e Fumo,1.0,79.736856,NaN,79.736856,79.736856,79.736856,79.736856,79.736856
Bolsas de Valores/Mercadorias e Futuros,1.0,9.921251,NaN,9.921251,9.921251,9.921251,9.921251,9.921251
Brinquedos e Lazer,3.0,1.702028,2.218559,0.160572,0.430670,0.700768,2.472755,4.244743
Comunicação e Informática,7.0,2.037814,1.611519,0.326589,0.975548,1.307844,3.091070,4.497028
Comércio (Atacado e Varejo),22.0,15.213687,34.877104,0.160707,0.967264,2.785684,12.401323,162.947000
"Construção Civil, Mat. Constr. e Decoração",30.0,1.679004,2.049809,0.002583,0.265317,1.035179,2.314096,7.429915
Educação,3.0,2.620554,0.991232,1.830934,2.064361,2.297787,3.015364,3.732940
